In [0]:
path = "abfss://source-adls@adlsmewalth.dfs.core.windows.net/ecommerce_product_catalog.xml"

   
This section ingests the XML product catalog incrementally with Auto Loader and writes the staged files, schema tracking, checkpoints, and Delta data directly to ADLS using `abfss://` paths.

Because Auto Loader watches directories, the source XML file is first staged into an ADLS directory and then ingested incrementally.

* `ekart_dt.bronze.bronze_product_catalog_xml_raw`: raw XML records parsed at the `category` level and stored in ADLS
* `ekart_dt.bronze.bronze_product_catalog_xml_flat`: flattened product and variant records stored in ADLS

The stream uses `availableNow=True`, so each run processes newly discovered files and then stops.

In [0]:
from pyspark.sql.functions import col, current_timestamp, explode, explode_outer

source_file_path = path

catalog_name = "ekart_dt"
bronze_schema = "bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")

bronze_raw_table = f"{catalog_name}.{bronze_schema}.bronze_product_catalog_xml_raw"
bronze_flat_table = f"{catalog_name}.{bronze_schema}.bronze_product_catalog_xml_flat"

adls_base_path = "abfss://source-adls@adlsmewalth.dfs.core.windows.net/bronze/xml_product_catalog"
staging_path = f"{adls_base_path}/input"
schema_location = f"{adls_base_path}/schema"
bronze_checkpoint = f"{adls_base_path}/checkpoints/bronze_raw"
bronze_raw_table_path = f"{adls_base_path}/tables/bronze_product_catalog_xml_raw"
bronze_flat_table_path = f"{adls_base_path}/tables/bronze_product_catalog_xml_flat"

source_filename = source_file_path.rsplit("/", 1)[-1]
staged_file_path = f"{staging_path}/{source_filename}"

dbutils.fs.mkdirs(staging_path)
dbutils.fs.cp(source_file_path, staged_file_path, True)

print({
    "source_file_path": source_file_path,
    "staged_file_path": staged_file_path,
    "bronze_raw_table": bronze_raw_table,
    "bronze_flat_table": bronze_flat_table,
    "schema_location": schema_location,
    "bronze_checkpoint": bronze_checkpoint,
    "bronze_raw_table_path": bronze_raw_table_path,
    "bronze_flat_table_path": bronze_flat_table_path
})

In [0]:
raw_xml_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "xml")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rowTag", "category")
        .load(staging_path)
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_ts", current_timestamp())
)

raw_xml_query = (
    raw_xml_stream.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(bronze_raw_table_path)
)

raw_xml_query.awaitTermination()

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {bronze_raw_table}
    USING DELTA
    LOCATION '{bronze_raw_table_path}'
    """
)

print(f"Loaded raw XML into ADLS path {bronze_raw_table_path}")
print(f"Registered table: {bronze_raw_table}")
print(f"Raw row count: {spark.read.format('delta').load(bronze_raw_table_path).count()}")

In [0]:
bronze_source_df = (
    spark.read.format("delta")
        .load(bronze_raw_table_path)
        .filter(col("_rescued_data").isNull())
)

flat_df = (
    bronze_source_df
        .withColumn("subcategory", explode("subcategory"))
        .withColumn("product", explode("subcategory.product"))
        .withColumn("variant", explode_outer("product.variants.variant"))
        .select(
            col("_name").alias("category_name"),
            col("subcategory._name").alias("subcategory_name"),
            col("subcategory._count").alias("subcategory_product_count"),
            col("product._product_id").alias("product_id"),
            col("product._in_stock").alias("product_in_stock"),
            col("product.name").alias("product_name"),
            col("product.brand").alias("brand"),
            col("product.price._VALUE").alias("price_amount"),
            col("product.price._currency").alias("price_currency"),
            col("product.rating").alias("rating"),
            col("product.stock_quantity").alias("stock_quantity"),
            col("variant._sku").alias("sku"),
            col("variant.size").alias("variant_size"),
            col("variant.color").alias("variant_color"),
            col("variant._stock").alias("variant_stock"),
            col("source_file"),
            col("ingestion_ts")
        )
)

(
    flat_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_flat_table_path)
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {bronze_flat_table}
    USING DELTA
    LOCATION '{bronze_flat_table_path}'
    """
)

print(f"Loaded flattened XML into ADLS path {bronze_flat_table_path}")
print(f"Registered table: {bronze_flat_table}")
print(f"Flattened row count: {spark.read.format('delta').load(bronze_flat_table_path).count()}")

In [0]:
%sql
select *
from delta.`abfss://source-adls@adlsmewalth.dfs.core.windows.net/bronze/xml_product_catalog/tables/bronze_product_catalog_xml_flat`